### Loading Pretrained BERT and ViT Models

In this step, I load the pretrained BERT and Vision Transformer (ViT) models. BERT will be used to extract text features from the questions and answer options, while ViT will extract image features. Both models will remain frozen initially to reduce training time and computational cost.

In [3]:
import torch
import torch.nn as nn
import numpy as np

from transformers import AutoModel
from datasets import load_from_disk

# Select device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

# Load preprocessed dataset
processed_dataset = load_from_disk(
    "../data/processed/scienceqa_preprocessed"
)

# Model names
BERT_MODEL = "bert-base-uncased"
VIT_MODEL = "google/vit-base-patch16-224"

# Load pretrained models
bert = AutoModel.from_pretrained(BERT_MODEL)
vit = AutoModel.from_pretrained(VIT_MODEL)

# Freeze both models
for param in bert.parameters():
    param.requires_grad = False

for param in vit.parameters():
    param.requires_grad = False

bert = bert.to(device)
vit = vit.to(device)

bert.eval()
vit.eval()

print("BERT and ViT loaded successfully.")

Using device: mps


W1111 17:53:44.598000 11757 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT and ViT loaded successfully.


### Fixing MPS Attention Compatibility

The default attention implementation caused a compatibility issue with Apple's MPS device. I changed the attention implementation to eager attention to avoid this issue while keeping the pretrained models frozen.

In [4]:
# Use eager attention instead of SDPA
bert = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze both models
for param in bert.parameters():
    param.requires_grad = False

for param in vit.parameters():
    param.requires_grad = False

bert = bert.to(device)
vit = vit.to(device)

bert.eval()
vit.eval()

print("BERT and ViT reloaded with eager attention.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT and ViT reloaded with eager attention.


### Checking Feature Dimensions

This step checks the output dimensions of BERT and ViT using one sample from the dataset. I will use these features later to combine the text and image information in the early fusion model.

In [5]:
sample = processed_dataset["train"][0]

# Prepare text inputs
input_ids = torch.tensor(
    sample["input_ids"], dtype=torch.long
).unsqueeze(0).to(device)

attention_mask = torch.tensor(
    sample["attention_mask"], dtype=torch.long
).unsqueeze(0).to(device)

# Prepare image input
pixel_values = torch.tensor(
    sample["pixel_values"], dtype=torch.float32
).unsqueeze(0).to(device)

# Extract features
with torch.no_grad():

    # BERT features for each answer option
    batch_size, num_choices, seq_length = input_ids.shape

    bert_output = bert(
        input_ids=input_ids.view(-1, seq_length),
        attention_mask=attention_mask.view(-1, seq_length)
    )

    text_features = bert_output.last_hidden_state[:, 0, :]

    # ViT image features
    vit_output = vit(pixel_values=pixel_values)

    image_features = vit_output.last_hidden_state[:, 0, :]

print("Text feature shape:", text_features.shape)
print("Image feature shape:", image_features.shape)

Text feature shape: torch.Size([5, 768])
Image feature shape: torch.Size([1, 768])


### Building the Early Fusion Model

In this step, I combine the text features extracted by BERT and the image features extracted by ViT. The two feature representations are concatenated and passed through a classification layer to score each answer option. Both pretrained models remain frozen, while the fusion layers are trained.

In [11]:
class EarlyFusionModel(nn.Module):
    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        # Freeze pretrained models
        for param in self.bert.parameters():
            param.requires_grad = False

        for param in self.vit.parameters():
            param.requires_grad = False

        # Fusion classifier
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, input_ids, attention_mask, pixel_values, option_mask):

        batch_size, num_choices, seq_length = input_ids.shape

        input_ids = input_ids.view(-1, seq_length)
        attention_mask = attention_mask.view(-1, seq_length)

        valid_indices = option_mask.view(-1).nonzero(as_tuple=True)[0]

        # Extract text features
        with torch.no_grad():
            text_output = self.bert(
                input_ids=input_ids[valid_indices],
                attention_mask=attention_mask[valid_indices]
            )

            text_features = text_output.last_hidden_state[:, 0, :]

            # Extract image features
            image_output = self.vit(pixel_values=pixel_values)
            image_features = image_output.last_hidden_state[:, 0, :]

        # Repeat image features for each valid answer option
        image_features = image_features.repeat_interleave(
            num_choices, dim=0
        )[valid_indices]

        # Concatenate text and image features
        combined_features = torch.cat(
            (text_features, image_features), dim=1
        )

        # Calculate answer scores
        valid_scores = self.fusion(combined_features).squeeze(-1)

        scores = torch.full(
            (batch_size * num_choices,),
            float("-inf"),
            device=input_ids.device
        )

        scores[valid_indices] = valid_scores

        return scores.view(batch_size, num_choices)


early_fusion_model = EarlyFusionModel(bert, vit).to(device)

print("Early Fusion model created successfully.")

Early Fusion model created successfully.


### Preparing DataLoaders for Early Fusion

This step prepares the training, validation and test datasets for the early fusion model. Each batch contains the question, answer options, and corresponding images so that BERT and ViT can process both modalities together.

In [12]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

# Load preprocessed dataset
processed_dataset = load_from_disk(
    "../data/processed/scienceqa_preprocessed"
)

class EarlyFusionDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        return {
            "input_ids": torch.tensor(sample["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(sample["attention_mask"], dtype=torch.long),
            "option_mask": torch.tensor(sample["option_mask"], dtype=torch.bool),
            "pixel_values": torch.tensor(sample["pixel_values"], dtype=torch.float32),
            "labels": torch.tensor(sample["labels"], dtype=torch.long)
        }

# Create datasets
train_dataset = EarlyFusionDataset(processed_dataset["train"])
val_dataset = EarlyFusionDataset(processed_dataset["validation"])
test_dataset = EarlyFusionDataset(processed_dataset["test"])

# Create DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 442
Validation batches: 168
Test batches: 253


Model testing

In [13]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
option_mask = batch["option_mask"].to(device)
pixel_values = batch["pixel_values"].to(device)

with torch.no_grad():
    logits = early_fusion_model(
        input_ids,
        attention_mask,
        pixel_values,
        option_mask
    )

print("Input text shape:", input_ids.shape)
print("Input image shape:", pixel_values.shape)
print("Output shape:", logits.shape)
print("Sample logits:", logits[0])

Input text shape: torch.Size([8, 5, 128])
Input image shape: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 5])
Sample logits: tensor([0.6321, 0.2488, 0.2027, 0.5714,   -inf], device='mps:0')


### Setting Up Early Fusion Training

This cell defines the cross-entropy loss function and AdamW optimizer. Only the fusion classifier will be trained, while the pretrained BERT and ViT models remain frozen.

In [14]:
early_fusion_model = EarlyFusionModel(bert, vit).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    early_fusion_model.fusion.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

print("Early Fusion model and optimizer recreated.")

Early Fusion model and optimizer recreated.


### Training and Validation Functions

These functions train the early fusion model and evaluate its performance using the validation dataset. During training, the fusion classifier learns to combine image and text features to predict the correct answer. BERT and ViT remain frozen.

In [15]:
from tqdm.auto import tqdm
import copy

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        option_mask = batch["option_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids,
            attention_mask,
            pixel_values,
            option_mask
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            option_mask = batch["option_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask,
                pixel_values,
                option_mask
            )

            loss = criterion(logits, labels)

            total_loss += loss.item()

            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

### Training the Early Fusion Model

The model is trained for a maximum of 10 epochs with early stopping patience of 4. After each epoch, training and validation loss and accuracy are recorded. The model weights with the lowest validation loss are saved and restored after training.

In [ ]:
# EPOCHS = 10
# PATIENCE = 4

# best_val_loss = float("inf")
# best_epoch = 0
# epochs_without_improvement = 0
# best_model_state = None

# history_early = {
#     "train_loss": [],
#     "train_accuracy": [],
#     "val_loss": [],
#     "val_accuracy": []
# }

# for epoch in range(EPOCHS):

#     print(f"\nEpoch {epoch + 1}/{EPOCHS}")

#     train_loss, train_acc = train_one_epoch(
#         early_fusion_model,
#         train_loader,
#         optimizer,
#         criterion
#     )

#     val_loss, val_acc = evaluate(
#         early_fusion_model,
#         val_loader,
#         criterion
#     )

#     history_early["train_loss"].append(train_loss)
#     history_early["train_accuracy"].append(train_acc)
#     history_early["val_loss"].append(val_loss)
#     history_early["val_accuracy"].append(val_acc)

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         best_epoch = epoch + 1
#         epochs_without_improvement = 0

#         best_model_state = copy.deepcopy(
#             early_fusion_model.state_dict()
#         )
#     else:
#         epochs_without_improvement += 1

#     print(
#         f"Train Loss: {train_loss:.4f} | "
#         f"Train Accuracy: {train_acc:.4f} | "
#         f"Val Loss: {val_loss:.4f} | "
#         f"Val Accuracy: {val_acc:.4f}"
#     )

#     print(
#         f"Best Val Loss: {best_val_loss:.4f} "
#         f"(Epoch {best_epoch})"
#     )

#     if epochs_without_improvement >= PATIENCE:
#         print(f"\nEarly stopping at epoch {epoch + 1}.")
#         break

# early_fusion_model.load_state_dict(best_model_state)

# print("\nTraining completed.")
# print("Best epoch:", best_epoch)
# print("Best validation loss:", round(best_val_loss, 4))


Epoch 1/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Validation:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 1.2240 | Train Accuracy: 0.3382 | Val Loss: 1.1384 | Val Accuracy: 0.4587
Best Val Loss: 1.1384 (Epoch 1)

Epoch 2/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Validation:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 1.1975 | Train Accuracy: 0.3300 | Val Loss: 1.1282 | Val Accuracy: 0.4661
Best Val Loss: 1.1282 (Epoch 2)

Epoch 3/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [22]:
test_loss, test_accuracy = evaluate(
    early_fusion_model,
    test_loader,
    criterion
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Validation:   0%|          | 0/253 [00:00<?, ?it/s]

Test Loss: 0.9830
Test Accuracy: 0.4963


### Experiment 1 Early Fusion Test Results

The Early Fusion model achieved a test accuracy of 49.63% and a test loss of 0.9830. Compared with the text-only BERT baseline, its test accuracy was lower by 7.14 percentage points.

This indicates that simple concatenation of frozen BERT and ViT features did not improve performance. One possible reason is that the frozen image and text representations were not sufficiently adapted to the ScienceQA task.

## Experiment 2: Early Fusion V2 – Fine-tuned BERT

In this experiment, I will improve the previous early fusion model by fine-tuning the last four layers of BERT while keeping ViT frozen. The aim is to allow BERT to better understand the ScienceQA questions while retaining the pretrained image features.

In [23]:
# Reload pretrained models
bert_v2 = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit_v2 = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze all BERT layers
for param in bert_v2.parameters():
    param.requires_grad = False

# Unfreeze last four BERT layers
for layer in bert_v2.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Freeze ViT
for param in vit_v2.parameters():
    param.requires_grad = False

bert_v2 = bert_v2.to(device)
vit_v2 = vit_v2.to(device)

print("BERT V2 and ViT V2 loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT V2 and ViT V2 loaded.


### Building Early Fusion V2

This model combines text features from fine-tuned BERT and image features from frozen ViT. The last four BERT layers are trainable, allowing the model to adapt to ScienceQA. The extracted text and image features are concatenated and passed through a classifier to predict the correct answer.

In [24]:
class EarlyFusionModelV2(nn.Module):
    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        # Fusion classifier
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def train(self, mode=True):
        super().train(mode)

        # Keep frozen ViT in evaluation mode
        self.vit.eval()

        return self

    def forward(self, input_ids, attention_mask, pixel_values, option_mask):

        batch_size, num_choices, seq_length = input_ids.shape

        input_ids = input_ids.view(-1, seq_length)
        attention_mask = attention_mask.view(-1, seq_length)

        valid_indices = option_mask.view(-1).nonzero(as_tuple=True)[0]

        # Extract text features
        text_output = self.bert(
            input_ids=input_ids[valid_indices],
            attention_mask=attention_mask[valid_indices]
        )

        text_features = text_output.last_hidden_state[:, 0, :]

        # Extract image features
        with torch.no_grad():
            image_output = self.vit(pixel_values=pixel_values)
            image_features = image_output.last_hidden_state[:, 0, :]

        # Repeat image features for each valid answer option
        image_features = image_features.repeat_interleave(
            num_choices, dim=0
        )[valid_indices]

        # Combine text and image features
        combined_features = torch.cat(
            (text_features, image_features), dim=1
        )

        valid_scores = self.fusion(combined_features).squeeze(-1)

        scores = torch.full(
            (batch_size * num_choices,),
            float("-inf"),
            device=input_ids.device
        )

        scores[valid_indices] = valid_scores

        return scores.view(batch_size, num_choices)


early_fusion_v2 = EarlyFusionModelV2(
    bert_v2, vit_v2
).to(device)

print("Early Fusion V2 created successfully.")

Early Fusion V2 created successfully.


### Configuring the Optimizer

This cell configures the AdamW optimizer with separate learning rates. A smaller learning rate is used for BERT because it is pretrained, while a higher learning rate is used for the newly created fusion classifier.

In [25]:
criterion_v2 = nn.CrossEntropyLoss()

optimizer_v2 = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in early_fusion_v2.bert.parameters()
                if p.requires_grad
            ],
            "lr": 2e-5
        },
        {
            "params": early_fusion_v2.fusion.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=0.01
)

print("V2 optimizer configured.")

V2 optimizer configured.


I define separate training and evaluation functions for Early Fusion V2. The model learns through the fusion classifier and the final four BERT layers, while the ViT image encoder remains frozen.

In [26]:
from tqdm.auto import tqdm
import torch
import torch.nn as nn

def train_one_epoch_v2(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training V2")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        option_mask = batch["option_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask,
            pixel_values,
            option_mask
        )

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total


def evaluate_v2(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            option_mask = batch["option_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids,
                attention_mask,
                pixel_values,
                option_mask
            )

            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [28]:
import copy

EPOCHS_V2 = 15
PATIENCE_V2 = 4

history_v2 = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss_v2 = float("inf")
best_epoch_v2 = 0
patience_counter_v2 = 0
best_model_weights_v2 = None

for epoch in range(EPOCHS_V2):
    print(f"\nEpoch {epoch + 1}/{EPOCHS_V2}")

    train_loss, train_accuracy = train_one_epoch_v2(
        early_fusion_v2,
        train_loader,
        optimizer_v2,
        criterion_v2
    )

    val_loss, val_accuracy = evaluate_v2(
        early_fusion_v2,
        val_loader,
        criterion_v2
    )

    history_v2["train_loss"].append(train_loss)
    history_v2["train_accuracy"].append(train_accuracy)
    history_v2["val_loss"].append(val_loss)
    history_v2["val_accuracy"].append(val_accuracy)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_accuracy:.4f}")

    if val_loss < best_val_loss_v2:
        best_val_loss_v2 = val_loss
        best_epoch_v2 = epoch + 1
        patience_counter_v2 = 0
        best_model_weights_v2 = copy.deepcopy(early_fusion_v2.state_dict())

        print("Best model updated.")

    else:
        patience_counter_v2 += 1
        print(f"No improvement: {patience_counter_v2}/{PATIENCE_V2}")

    if patience_counter_v2 >= PATIENCE_V2:
        print("Early stopping triggered.")
        break

early_fusion_v2.load_state_dict(best_model_weights_v2)

print("\nV2 Training completed.")
print(f"Best epoch: {best_epoch_v2}")
print(f"Best validation loss: {best_val_loss_v2:.4f}")


Epoch 1/15


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.6519 | Train Accuracy: 0.6714
Val Loss:   0.6646 | Val Accuracy:   0.6642
Best model updated.

Epoch 2/15


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.6199 | Train Accuracy: 0.6864
Val Loss:   0.6694 | Val Accuracy:   0.6835
No improvement: 1/4

Epoch 3/15


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.6096 | Train Accuracy: 0.6890
Val Loss:   0.6949 | Val Accuracy:   0.6880
No improvement: 2/4

Epoch 4/15


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.5854 | Train Accuracy: 0.7023
Val Loss:   0.7113 | Val Accuracy:   0.6947
No improvement: 3/4

Epoch 5/15


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.5617 | Train Accuracy: 0.7173
Val Loss:   0.7549 | Val Accuracy:   0.6992
No improvement: 4/4
Early stopping triggered.

V2 Training completed.
Best epoch: 1
Best validation loss: 0.6646


In [29]:
test_loss_v2, test_accuracy_v2 = evaluate_v2(
    early_fusion_v2,
    test_loader,
    criterion_v2
)

print("Early Fusion V2 Test Results")
print(f"Test Loss: {test_loss_v2:.4f}")
print(f"Test Accuracy: {test_accuracy_v2:.4f}")
print(f"Test Accuracy (%): {test_accuracy_v2 * 100:.2f}%")

Early Fusion V2 Test Results
Test Loss: 0.7826
Test Accuracy: 0.5910
Test Accuracy (%): 59.10%


Preserve V2

In [30]:
import os
import torch
import json

save_dir_v2 = "../results/early_fusion_v2"
os.makedirs(save_dir_v2, exist_ok=True)

torch.save(
    early_fusion_v2.state_dict(),
    os.path.join(save_dir_v2, "best_model.pth")
)

with open(os.path.join(save_dir_v2, "training_history.json"), "w") as f:
    json.dump(history_v2, f, indent=4)

print("V2 checkpoint saved successfully.")
print(f"V2 test accuracy: {test_accuracy_v2 * 100:.2f}%")

V2 checkpoint saved successfully.
V2 test accuracy: 59.10%


## Experiment 3: Early Fusion v3
I initialise separate BERT and ViT encoders for Early Fusion V3. I fine-tune the last four BERT layers and the last two ViT layers while keeping the remaining layers frozen.

In [17]:
from transformers import AutoModel

# Load separate V3 encoders
bert_v3 = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit_v3 = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze all BERT parameters
for param in bert_v3.parameters():
    param.requires_grad = False

# Unfreeze last 4 BERT layers
for layer in bert_v3.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Freeze all ViT parameters
for param in vit_v3.parameters():
    param.requires_grad = False

# Unfreeze last 2 ViT layers
for layer in vit_v3.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Move models to device
bert_v3 = bert_v3.to(device)
vit_v3 = vit_v3.to(device)

print("Early Fusion V3 encoders loaded successfully.")
print("BERT trainable layers: 4")
print("ViT trainable layers: 2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Early Fusion V3 encoders loaded successfully.
BERT trainable layers: 4
ViT trainable layers: 2


In [18]:
bert_trainable_v3 = sum(
    p.numel() for p in bert_v3.parameters() if p.requires_grad
)

vit_trainable_v3 = sum(
    p.numel() for p in vit_v3.parameters() if p.requires_grad
)

print(f"BERT trainable parameters: {bert_trainable_v3:,}")
print(f"ViT trainable parameters: {vit_trainable_v3:,}")
print(f"Total trainable encoder parameters: {bert_trainable_v3 + vit_trainable_v3:,}")

BERT trainable parameters: 28,351,488
ViT trainable parameters: 14,175,744
Total trainable encoder parameters: 42,527,232


Test V3 feature extraction

In [19]:
import torch

# Get one image batch
sample_batch = next(iter(train_loader))

sample_images = sample_batch["pixel_values"].to(device)

# Extract ViT features
vit_v3.eval()

with torch.no_grad():
    vit_outputs = vit_v3(pixel_values=sample_images)

    image_tokens = vit_outputs.last_hidden_state

    cls_features = image_tokens[:, 0, :]
    patch_features = image_tokens[:, 1:, :].mean(dim=1)

    combined_image_features = torch.cat(
        [cls_features, patch_features],
        dim=1
    )

print("Image tokens:", image_tokens.shape)
print("CLS features:", cls_features.shape)
print("Mean patch features:", patch_features.shape)
print("Combined image features:", combined_image_features.shape)

Image tokens: torch.Size([8, 197, 768])
CLS features: torch.Size([8, 768])
Mean patch features: torch.Size([8, 768])
Combined image features: torch.Size([8, 1536])


I create the Early Fusion V3 architecture. I combine BERT text embeddings with ViT global and patch-level image features. A deeper classifier with LayerNorm and Dropout predicts the correct answer.

In [20]:
import torch
import torch.nn as nn


class EarlyFusionModelV3(nn.Module):

    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        # Feature dimensions:
        # BERT: 768
        # ViT CLS + mean patches: 1536
        fusion_input_size = 768 + 1536

        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, 1)
        )

    def train(self, mode=True):
        super().train(mode)
        return self

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        option_mask
    ):

        batch_size, num_choices, seq_length = input_ids.shape

        # Flatten answer choices for BERT
        flat_input_ids = input_ids.view(
            batch_size * num_choices, seq_length
        )

        flat_attention_mask = attention_mask.view(
            batch_size * num_choices, seq_length
        )

        # Extract text features
        text_outputs = self.bert(
            input_ids=flat_input_ids,
            attention_mask=flat_attention_mask
        )

        text_features = text_outputs.last_hidden_state[:, 0, :]

        # Extract image features
        # Each image is processed once per question
        image_outputs = self.vit(
            pixel_values=pixel_values
        )

        image_tokens = image_outputs.last_hidden_state

        cls_features = image_tokens[:, 0, :]
        patch_features = image_tokens[:, 1:, :].mean(dim=1)

        image_features = torch.cat(
            [cls_features, patch_features],
            dim=1
        )

        # Repeat image features for each answer choice
        image_features = image_features.repeat_interleave(
            num_choices, dim=0
        )

        # Combine text and image features
        combined_features = torch.cat(
            [text_features, image_features],
            dim=1
        )

        # Predict answer scores
        scores = self.fusion(combined_features)

        scores = scores.view(batch_size, num_choices)

        # Mask padded choices
        scores = scores.masked_fill(
            option_mask == 0,
            float("-inf")
        )

        return scores

In [21]:
early_fusion_v3 = EarlyFusionModelV3(
    bert=bert_v3,
    vit=vit_v3
).to(device)

print("Early Fusion V3 initialised.")

Early Fusion V3 initialised.


Testing Forward pass

In [22]:
sample_batch = next(iter(train_loader))

early_fusion_v3.eval()

with torch.no_grad():
    sample_outputs = early_fusion_v3(
        input_ids=sample_batch["input_ids"].to(device),
        attention_mask=sample_batch["attention_mask"].to(device),
        pixel_values=sample_batch["pixel_values"].to(device),
        option_mask=sample_batch["option_mask"].to(device)
    )

print("Input batch:", sample_batch["input_ids"].shape)
print("Output scores:", sample_outputs.shape)
print("Sample predictions:", sample_outputs.argmax(dim=1))

Input batch: torch.Size([8, 5, 128])
Output scores: torch.Size([8, 5])
Sample predictions: tensor([0, 0, 0, 0, 2, 0, 0, 1], device='mps:0')


In [23]:
import torch.nn as nn

criterion_v3 = nn.CrossEntropyLoss()

optimizer_v3 = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in early_fusion_v3.bert.parameters()
                if p.requires_grad
            ],
            "lr": 2e-5
        },
        {
            "params": [
                p for p in early_fusion_v3.vit.parameters()
                if p.requires_grad
            ],
            "lr": 5e-6
        },
        {
            "params": early_fusion_v3.fusion.parameters(),
            "lr": 5e-4
        }
    ],
    weight_decay=0.01
)

print("V3 optimiser configured.")

V3 optimiser configured.


In [24]:
from transformers import get_linear_schedule_with_warmup

EPOCHS_V3 = 8

total_training_steps_v3 = len(train_loader) * EPOCHS_V3

warmup_steps_v3 = int(0.1 * total_training_steps_v3)

scheduler_v3 = get_linear_schedule_with_warmup(
    optimizer_v3,
    num_warmup_steps=warmup_steps_v3,
    num_training_steps=total_training_steps_v3
)

print("Total training steps:", total_training_steps_v3)
print("Warmup steps:", warmup_steps_v3)

Total training steps: 3536
Warmup steps: 353


### V3 training and evaluation functions
I define separate training and evaluation functions for Early Fusion V3. During training, I update BERT, ViT and the fusion classifier while monitoring validation performance.

In [25]:
from tqdm.auto import tqdm
import torch
import torch.nn as nn


def train_one_epoch_v3(model, loader, optimizer, scheduler, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training V3")

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        option_mask = batch["option_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask,
            pixel_values,
            option_mask
        )

        loss = criterion(outputs, labels)

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total


def evaluate_v3(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            option_mask = batch["option_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids,
                attention_mask,
                pixel_values,
                option_mask
            )

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


print("V3 training and evaluation functions ready.")

V3 training and evaluation functions ready.


In [26]:
import copy

PATIENCE_V3 = 4

history_v3 = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss_v3 = float("inf")
best_epoch_v3 = 0
patience_counter_v3 = 0
best_model_weights_v3 = None

for epoch in range(EPOCHS_V3):

    print(f"\nEpoch {epoch + 1}/{EPOCHS_V3}")

    train_loss, train_accuracy = train_one_epoch_v3(
        early_fusion_v3,
        train_loader,
        optimizer_v3,
        scheduler_v3,
        criterion_v3
    )

    val_loss, val_accuracy = evaluate_v3(
        early_fusion_v3,
        val_loader,
        criterion_v3
    )

    history_v3["train_loss"].append(train_loss)
    history_v3["train_accuracy"].append(train_accuracy)
    history_v3["val_loss"].append(val_loss)
    history_v3["val_accuracy"].append(val_accuracy)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_accuracy:.4f}")

    if val_loss < best_val_loss_v3:

        best_val_loss_v3 = val_loss
        best_epoch_v3 = epoch + 1
        patience_counter_v3 = 0

        best_model_weights_v3 = copy.deepcopy(
            early_fusion_v3.state_dict()
        )

        print("Best model updated.")

    else:
        patience_counter_v3 += 1
        print(f"No improvement: {patience_counter_v3}/{PATIENCE_V3}")

    if patience_counter_v3 >= PATIENCE_V3:
        print("Early stopping triggered.")
        break

# Restore best checkpoint
early_fusion_v3.load_state_dict(best_model_weights_v3)

print("\nEarly Fusion V3 training completed.")
print(f"Best epoch: {best_epoch_v3}")
print(f"Best validation loss: {best_val_loss_v3:.4f}")


Epoch 1/8


Training V3:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 1.1573 | Train Accuracy: 0.3705
Val Loss:   0.8632 | Val Accuracy:   0.5763
Best model updated.

Epoch 2/8


Training V3:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.8976 | Train Accuracy: 0.5428
Val Loss:   0.7118 | Val Accuracy:   0.6180
Best model updated.

Epoch 3/8


Training V3:   0%|          | 0/442 [00:00<?, ?it/s]

... (7 outputs left out)

Training V3:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.6301 | Train Accuracy: 0.6875
Val Loss:   0.7019 | Val Accuracy:   0.6828
No improvement: 1/4

Epoch 8/8


Training V3:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.6079 | Train Accuracy: 0.6864
Val Loss:   0.6985 | Val Accuracy:   0.6865
No improvement: 2/4

Early Fusion V3 training completed.
Best epoch: 6
Best validation loss: 0.6701


In [27]:
test_loss_v3, test_accuracy_v3 = evaluate_v3(
    early_fusion_v3,
    test_loader,
    criterion_v3
)

print(f"V3 Test Loss: {test_loss_v3:.4f}")
print(f"V3 Test Accuracy: {test_accuracy_v3:.4f}")
print(f"V3 Test Accuracy: {test_accuracy_v3 * 100:.2f}%")

V3 Test Loss: 0.8285
V3 Test Accuracy: 0.5845
V3 Test Accuracy: 58.45%


In [28]:
import os
import json
import torch
import pandas as pd
import numpy as np
import shutil

# ==========================================
# 1. CREATE RESULTS DIRECTORY
# ==========================================

SAVE_DIR = "results/early_fusion_v3"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# 2. SAVE BEST MODEL CHECKPOINT
# ==========================================

# IMPORTANT:
# Use the actual path where your training loop
# saved the best model at epoch 6.

SOURCE_CHECKPOINT = "results/early_fusion_v3/best_model.pth"

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR, "best_model.pth"
)

if os.path.abspath(SOURCE_CHECKPOINT) != os.path.abspath(CHECKPOINT_PATH):
    if os.path.exists(SOURCE_CHECKPOINT):
        shutil.copy2(SOURCE_CHECKPOINT, CHECKPOINT_PATH)
    else:
        raise FileNotFoundError(
            f"Best checkpoint not found: {SOURCE_CHECKPOINT}. "
            "Check the checkpoint path used in your training loop."
        )

print("Best checkpoint saved:", CHECKPOINT_PATH)

# ==========================================
# 3. SAVE TRAINING HISTORY
# ==========================================

history_v3 = {
    "train_loss": [
        1.1573, 0.8976, 0.7716, 0.7046,
        0.6704, 0.6463, 0.6301, 0.6079
    ],
    "train_accuracy": [
        0.3705, 0.5428, 0.6037, 0.6394,
        0.6618, 0.6728, 0.6875, 0.6864
    ],
    "val_loss": [
        0.8632, 0.7118, 0.6747, 0.6898,
        0.6916, 0.6701, 0.7019, 0.6985
    ],
    "val_accuracy": [
        0.5763, 0.6180, 0.6530, 0.6605,
        0.6694, 0.6850, 0.6828, 0.6865
    ]
}

with open(
    os.path.join(SAVE_DIR, "training_history.json"), "w"
) as f:
    json.dump(history_v3, f, indent=4)

pd.DataFrame(history_v3).to_csv(
    os.path.join(SAVE_DIR, "training_history.csv"),
    index=False
)

print("Training history saved.")

# ==========================================
# 4. SAVE MODEL CONFIGURATION
# ==========================================

metadata = {
    "model_name": "Early Fusion V3",
    "bert_model": "bert-base-uncased",
    "vit_model": "google/vit-base-patch16-224",
    "bert_trainable_layers": 4,
    "vit_trainable_layers": 2,
    "visual_features": "CLS + mean-pooled image patches",
    "fusion": "Feature concatenation",
    "learning_rates": {
        "bert": 2e-5,
        "vit": 5e-6,
        "fusion": 5e-4
    },
    "batch_size": 8,
    "maximum_epochs": 8,
    "best_epoch": 6,
    "best_validation_loss": 0.6701,
    "best_epoch_validation_accuracy": 0.6850,
    "early_stopping_patience": 4,
    "weight_decay": 0.01
}

with open(
    os.path.join(SAVE_DIR, "metadata.json"), "w"
) as f:
    json.dump(metadata, f, indent=4)

print("Model metadata saved.")

# ==========================================
# 5. VERIFY SAVED FILES
# ==========================================

print("\nSAVED FILES")
print("=" * 40)

for filename in os.listdir(SAVE_DIR):
    path = os.path.join(SAVE_DIR, filename)
    print(f"{filename}: {os.path.getsize(path) / 1024:.2f} KB")

print("\nV3 checkpoint preparation completed!")

Best checkpoint saved: results/early_fusion_v3/best_model.pth
Training history saved.
Model metadata saved.

SAVED FILES
metadata.json: 0.58 KB
training_history.csv: 0.26 KB
training_history.json: 0.60 KB

V3 checkpoint preparation completed!


In [32]:
print("Model exists:", "early_fusion_v3" in globals())
print("Best weights exist:", "best_model_weights_v3" in globals())
print("Best epoch:", globals().get("best_epoch_v3", "Not found"))

Model exists: True
Best weights exist: True
Best epoch: 6


In [33]:

import os
import torch

# Create the correct results folder
save_dir = "../results/early_fusion_v3"
os.makedirs(save_dir, exist_ok=True)

# Save the best epoch 6 weights
checkpoint_path = os.path.join(save_dir, "best_model.pth")

torch.save(
    best_model_weights_v3,
    checkpoint_path
)

print("V3 checkpoint saved successfully!")
print("Best epoch:", best_epoch_v3)
print("Checkpoint:", checkpoint_path)
print(f"File size: {os.path.getsize(checkpoint_path) / (1024 * 1024):.2f} MB")


V3 checkpoint saved successfully!
Best epoch: 6
Checkpoint: ../results/early_fusion_v3/best_model.pth
File size: 752.35 MB
